# Regional Risk Matrix: Frequency vs. Lethality

This notebook generates the **Regional Risk Matrix**, a quadrant-based visualization used to differentiate between high-frequency instability and high-intensity combat zones. 

### Key Metrics:
1.  **Frequency (X-axis):** Total number of conflict events in a region.
2.  **Lethality / Severity Index (Y-axis):** Average fatalities per event ($Fatalities / Events$).
3.  **Baseline Lethality:** The national average lethality, used as a threshold to identify "Red Zones."

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import sys

# Add root directory to path
sys.path.append(os.path.abspath('..'))
from src.processing import clean_conflict_data

## 1. Data Loading and Aggregation

In [2]:
raw_path = '../raw_data_output.csv'
df = clean_conflict_data(pd.read_csv(raw_path))

# Aggregate by Region
risk_df = df.groupby('admin1').agg({
    'event_id_cnty': 'count', 
    'fatalities': 'sum'
}).rename(columns={'event_id_cnty': 'Frequency'})

# Calculate Lethality (Severity Index)
risk_df['Lethality'] = (risk_df['fatalities'] / risk_df['Frequency']).round(2)

# Calculate Baseline Lethality (National Mean)
baseline_lethality = risk_df['Lethality'].mean()

print(f"National Baseline Lethality: {baseline_lethality:.2f}")
risk_df.sort_values('Lethality', ascending=False).head()

National Baseline Lethality: 0.96


,Frequency,fatalities,Lethality
admin1,,,
Kayah,1773,3062,1.73
Kayin,3075,4642,1.51
Rakhine,3908,5692,1.46
Chin,2291,3231,1.41
Magway,7848,9814,1.25


## 2. Generating the Risk Matrix

We plot the regions on a scatter plot. 
- **Size of bubble:** Total fatalities.
- **Color:** Intensity of lethality.
- **Dash Line:** The Baseline Lethality threshold.

In [3]:
fig = px.scatter(
    risk_df.reset_index(), 
    x="Frequency", 
    y="Lethality", 
    text="admin1", 
    size="fatalities", 
    color="Lethality",
    color_continuous_scale="Reds",
    title="Regional Risk Matrix (Frequency vs. Lethality)",
    labels={'Lethality': 'Lethality (Fatalities/Event)'},
    template="plotly_white",
    height=700
)

# Add the Baseline Lethality Line
fig.add_hline(
    y=baseline_lethality, 
    line_dash="dash", 
    line_color="red",
    annotation_text=f"Baseline Lethality ({baseline_lethality:.2f})", 
    annotation_position="bottom right"
)

fig.update_traces(textposition='top center')
fig.update_layout(
    xaxis_title="Conflict Frequency (Total Events)",
    yaxis_title="Regional Severity Index (Lethality)",
    coloraxis_showscale=False
)

fig.show()

## 3. Quadrant Interpretation

The baseline line divides the chart into two critical zones:

1.  **Upper-Right (Red Zone):** High Frequency + High Lethality. These regions (like **Sagaing**, **Kayah**) are the most critical for trauma-focused aid.
2.  **Upper-Left (High-Intensity Combat):** Low Frequency + High Lethality. These areas (like **Rakhine**) experience fewer but much more deadly engagements.
3.  **Lower-Right (Persistent Instability):** High Frequency + Low Lethality. Conflict is common but less deadly (e.g., urban protests or harassment).
4.  **Lower-Left (Low-Level Friction):** Below average on both scales.

In [4]:
# Identifying the Red Zone Regions
red_zones = risk_df[risk_df['Lethality'] > baseline_lethality].sort_values('Frequency', ascending=False)
print("Regions Above Baseline Lethality (High Risk):")
print(red_zones)

Regions Above Baseline Lethality (High Risk):
            Frequency  fatalities  Lethality
admin1                                      
Sagaing         23338       26552       1.14
Magway           7848        9814       1.25
Rakhine          3908        5692       1.46
Kayin            3075        4642       1.51
Bago-East        2914        3327       1.14
Chin             2291        3231       1.41
Shan-South       2114        2266       1.07
Kayah            1773        3062       1.73
Bago-West        1558        1660       1.07
